# 00. Adathalmaz Áttekintés és Minőségi Riport

**TDK Kutatási Téma**: a vizsgált terület lakóingatlan-piacának komplex térökonometriai, hedonikus és gépi tanulásos vizsgálata — különös tekintettel a vasúti infrastruktúra kettős (terhelő és értéknövelő) ingatlanpiaci hatására.

---

### 📖 A kutatás célja és módszertani alapvetése
A lakóingatlan-árakat nem csupán az épületek fizikai paraméterei (méret, szobaszám, állapot) határozzák meg, hanem az ingatlanok **térbeli elhelyezkedése és környezeti beágyazottsága** is. A kutatás modern adattudományi, térinformatikai (GIS) és ökonometriai eszközökkel számszerűsíti az áralakító erőket: a hálózati (gyalogos) távolságokat, a közösségi közlekedés (TOD) árfelhajtó hatását, a környezeti terhelések (például vasúti zaj) értékcsökkentő szerepét, valamint a közberuházások által generált értéknövekmény visszanyerési lehetőségeit (Land Value Capture).

### 🗂️ Az adathalmaz felépítése és megbízhatósága
Az elemzés fundamentuma egy nyílt piaci (kínálati) adatbázis, amelyet térinformatikai rétegekkel (OpenStreetMap POI-k, kötöttpályás hálózat) egészítünk ki. Minden megfigyeléshez nagyszámú változó tartozik: épületfizikai paraméterek, hálózati gyalogos távolságok, környezeti terhelési mutatók és intézményi ellátottsági indexek. Az adatok szigorú tisztítási folyamaton (adattisztítás, duplikációszűrés, outlier-kezelés, geokódolás) mennek keresztül az elemzés torzítatlansága érdekében.

**A fals precizitás kizárása:** koordinátát és hálózati távolságot kizárólag azokra a hirdetésekre számítunk, amelyeknél pontos (épületszintű) geokódolás áll rendelkezésre; a többi rekord térbeli mezője szigorúan üres marad (`minta_garantalt_pontos = 0`).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Adathalmaz betöltése
df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
print(f"Sikeresen betöltve: {len(df)} sor, {len(df.columns)} oszlop.")

### 1. Alapvető Adatbázis KPI Mutatók és a Piac Szerkezete

**📌 Mit mérünk:**
A vezérlőpult a terület validált sarokszámait foglalja össze: a teljes hirdetésállomány, az eladó és kiadó szegmens aránya, a garantáltan pontos koordinátájú minta nagysága és aránya, a medián négyzetméterár és az átlagos alapterületek.

**🔍 Tudományos interpretáció:**
1. **Tulajdoni szerkezet:** az eladó/kiadó arány a terület lakáspiaci funkcióját mutatja — a dominánsan eladó piac a saját tulajdonú otthonok, a magas bérleti arány a befektetési/tranzit zóna jelleg indikátora.
2. **Középértékek robusztussága:** az ingatlanárak eloszlása jellemzően jobbra ferde (log-normális), ezért az elemzésben a **mediánt** tekintjük a piac torzítatlan középértékének a számtani átlag helyett.
3. **Térbeli pontosság:** a precíz hálózati távolságméréshez és a térökonometriai súlymátrixokhoz kizárólag a garantáltan pontos almintát használjuk.

In [ ]:
n_total = len(df)
n_elado = len(df[df['listing_type'] == 'elado'])
n_kiado = len(df[df['listing_type'] == 'kiado'])
n_pontos = len(df[df['minta_garantalt_pontos'] == 1])
median_nm_ar = df[df['listing_type'] == 'elado']['nm_ar_huf'].median()
atlag_alapterulet = df['alapterulet_nm'].mean()

kpi_cards = [
    ("Összes Hirdetés", f"{n_total:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó Lakások", f"{n_elado:,} db".replace(',', ' '), f"{n_elado/n_total*100:.1f}% arány", "#2563eb"),
    ("Kiadó Lakások", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_total*100:.1f}% arány", "#d97706"),
    ("Garantált Pontos GIS", f"{n_pontos:,} db".replace(',', ' '), f"{n_pontos/n_total*100:.1f}% koordinátás", "#16a34a"),
    ("Eladó Medián Ár/m²", fmt_huf(median_nm_ar), "Kínálati fajlagos ár", "#9333ea"),
    ("Átlagos Alapterület", f"{atlag_alapterulet:.1f} m²", "Kínálati átlag", "#0891b2"),
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kínálati Megoszlás Térségi Szegmensek Szerint

**📌 Mit mutat az ábra:**
A diagram a terület belső városrészeinek kínálati súlyát mutatja, elkülönítve az eladó és kiadó állományt.

**🔍 Ok-okozati összefüggések:**
- **Koncentráció:** a magas hirdetésszámú zónák általában a sűrű beépítésű (lakótelepi) vagy intenzív fluktuációjú területek.
- **Funkcionális elkülönülés:** a bérleti piac aktivitása jellemzően a kiváló közlekedési kapcsolatú központokban és az egyetemi/irodai folyosók közelében a legmagasabb; a periférikus zónákban strukturálisan alacsonyabb.

In [ ]:
df_counts = df.groupby(['varosresz', 'listing_type']).size().reset_index(name='count')
df_counts['listing_nev'] = df_counts['listing_type'].map({'elado': 'Eladó', 'kiado': 'Kiadó'})

fig1 = px.bar(
    df_counts,
    x='varosresz',
    y='count',
    color='listing_nev',
    barmode='group',
    title='Hirdetések megoszlása városrészenként és típusonként',
    labels={'varosresz': 'Városrész', 'count': 'Hirdetések száma', 'listing_nev': 'Típus'},
    color_discrete_map={'Eladó': '#2563eb', 'Kiadó': '#f59e0b'},
    template=PLOTLY_TEMPLATE
)
fig1.update_layout(xaxis_tickangle=-30, height=450)
fig1.show()

### 3. Változók Kitöltöttsége (Adatminőségi Audit)

**📌 A minőségbiztosítás elvei:**
A vízszintes sávdiagram a változók adatelérhetőségét (kitöltöttségi arányát) mutatja 0–100%-ig.

**🔍 Miért kritikus ez az ökonometriában:**
- **Fundamentális változók:** az ár, a szobaszám és az alapterület teljes kitöltöttsége a hedonikus becslés alapfeltétele.
- **Információs aszimmetria:** a közepes kitöltöttségű paraméterek hiánya gyakran nem véletlenszerű — a rosszabb adottságú ingatlanok hirdetői hajlamosak elhallgatni ezeket.
- **Modellezési stratégia:** a térbeli modelleket csak a hiánytalan geokódolású mintán futtatjuk, biztosítva a szigorú geometriai topológiát.

In [ ]:
# 1. Teljes változó-metaadatbázis definiálása
valtozo_meta = [
    # Pénzügyi célváltozók
    ('price_huf', 'Kínálati vételár', 'Pénzügyi', 'HUF', 'Célváltozó (bruttó összeg)'),
    ('nm_ar_huf', 'Négyzetméterár', 'Pénzügyi', 'HUF/m²', 'Fajlagos célváltozó'),
    ('log_nm_ar', 'Fajlagos ár logaritmusa', 'Pénzügyi', 'ln(HUF/m²)', 'Ökonometriai regressziós cél'),
    ('ar_millio_ft', 'Vételár millió Ft-ban', 'Pénzügyi', 'M Ft', 'Leíró statisztikai mutató'),
    ('ar_ezer_ft_ho', 'Havi bérleti díj', 'Pénzügyi', 'ezer Ft/hó', 'Bérleti piac és Rent Gap'),
    # Fizikai és épületjellemzők
    ('alapterulet_nm', 'Alapterület', 'Fizikai', 'm²', 'Fizikai alaptulajdonság'),
    ('korrigalt_alapterulet_nm', 'Korrigált alapterület', 'Fizikai', 'm²', 'Fél erkéllyel súlyozva'),
    ('szobaszam_osszes', 'Összes szobaszám', 'Fizikai', 'db', 'Belső beosztási kontroll'),
    ('allapot', 'Műszaki állapot (szöveg)', 'Fizikai', 'kategória', 'Minőségi besorolás'),
    ('allapot_kod', 'Műszaki állapot index', 'Fizikai', '1-6 skála', 'Hedonikus minőségi rang'),
    ('is_panel', 'Panelszerkezet dummy', 'Fizikai', '0/1', 'Technológiai diszkont'),
    ('is_tegla', 'Tégla falazat dummy', 'Fizikai', '0/1', 'Hagyományos falazat'),
    ('has_lift', 'Lift megléte dummy', 'Fizikai', '0/1', 'Kényelmi felszereltség'),
    ('has_erkely', 'Erkély megléte dummy', 'Fizikai', '0/1', 'Kültéri kapcsolat prémiuma'),
    ('erkely_nm', 'Erkély mérete', 'Fizikai', 'm²', 'Kültéri felület nagysága'),
    ('emelet_szam', 'Emelet szintszám', 'Fizikai', 'szint', 'Függőleges elhelyezkedés'),
    ('epulet_kora_ev', 'Épület becsült kora', 'Fizikai', 'év', 'Amortizációs hatás'),
    ('epites_eve_kategoria', 'Építési korszak', 'Fizikai', 'korszak', 'Építészeti korcsoport'),
    # Hálózati közlekedési távolságok
    ('tavolsag_metro_halozati_m', 'Metróállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Gyalogos metróelérés (M2/M3)'),
    ('tavolsag_vasut_halozati_m', 'Vasútállomás hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Állomási elérhetőség (TOD)'),
    ('tavolsag_villamos_halozati_m', 'Villamosmegálló hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Felszíni kötöttpályás hálózat'),
    ('tavolsag_busz_halozati_m', 'Buszmegálló hálózati táv.', 'Hálózati elérhetőség', 'méter', 'Helyi buszhálózat elérése'),
    ('tavolsag_park_halozati_m', 'Park / zöldfelület hálózat', 'Hálózati elérhetőség', 'méter', 'Rekreációs zöldterület elérése'),
    # Légvonalbeli környezeti externáliák
    ('tavolsag_vasut_m', 'Vasúti pálya légvonal (zaj)', 'Környezeti externália', 'méter', 'Immissziós zaj- és rezgésterhelés'),
    ('vasut_zona', 'Vasúti immissziós sáv', 'Környezeti externália', 'kategória', 'A rögzített 0-150-300-500-1000-2000 m sáv'),
    ('is_vasut_immisszio_150m', 'Közvetlen immisszió (<150m)', 'Környezeti externália', '0/1', 'Immissziós zóna dummy'),
    ('tavolsag_kotottpalya_halozati_m', 'Kötöttpálya-index (hálózat)', 'Hálózati elérhetőség', 'méter', 'min(metró, vasút, villamos) hálózati táv'),
    ('tavolsag_metro_m', 'Metróvonal légvonal', 'Környezeti externália', 'méter', 'Légvonalbeli referencia táv'),
    ('tavolsag_park_m', 'Legközelebbi park légvonal', 'Környezeti externália', 'méter', 'Zöldfelületi közelség'),
    # Gyalogos izokrón dummyk
    ('metro_10p_seta', 'Metró 10p séta (750m)', 'Gyalogos izokrón', '0/1', 'Metró vonzáskörzet'),
    ('vasut_10p_seta', 'Vasút 10p séta (750m)', 'Gyalogos izokrón', '0/1', 'Állomás gyalogos elérhetőség'),
    ('villamos_10p_seta', 'Villamos 10p séta', 'Gyalogos izokrón', '0/1', 'Villamos vonzáskörzet'),
    ('park_10p_seta', 'Park 10p séta', 'Gyalogos izokrón', '0/1', 'Közeli zöldterületi ellátás'),
    # POI-sávok (hálózati elérhetőség, 0-375-750-1125 m)
    ('poi_5p_count', 'POI 5p (≤375m) hálózat', 'POI ellátottság', 'db', 'Közvetlen környezet szolgáltatásai'),
    ('poi_10p_count', 'POI 10p (≤750m) hálózat', 'POI ellátottság', 'db', 'Napi szükségletek elérhetősége'),
    ('poi_15p_count', 'POI 15p (≤1125m) hálózat', 'POI ellátottság', 'db', '15 perces város ellátottság'),
    # Térbeli GIS koordináták
    ('geokodolt_lat', 'WGS84 Szélesség (Lat)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('geokodolt_lon', 'WGS84 Hosszúság (Lon)', 'Térbeli GIS', 'fok', 'Térbeli pontkoordináta'),
    ('varosresz', 'Kőbányai Városrész', 'Térbeli GIS', 'név', 'Városrészi szegmens'),
    ('minta_garantalt_pontos', 'Pontos koordináta zászló', 'Térbeli GIS', '0/1', 'GIS szűrési jelző (pontos minta zászló)')
]

# 2. Kitöltöttségi statisztikák számítása
# A hálózati/POI változók strukturálisan csak a garantáltan pontos almintán definiáltak.
df_pontos = df[df['minta_garantalt_pontos'] == 1]
stats_rows = []
dict_rows = []
for col, hu_name, kat, unit, role in valtozo_meta:
    if col in df.columns:
        structural = (col.startswith('tavolsag_') or col.startswith('menetido_') or col.startswith('poi_'))
        base = df_pontos if structural else df
        n_valid = base[col].notna().sum()
        pct = (n_valid / len(base)) * 100
        note = ' (pontos almintán)' if structural else ''
        stats_rows.append({
            'Technikai oszlop': col,
            'Változó megnevezése': hu_name + note,
            'Kategória': kat,
            'Kitöltöttség (%)': round(pct, 1),
            'Érvényes N': n_valid
        })
        dict_rows.append({
            'Kategória': kat,
            'Változó neve': hu_name,
            'Technikai név': f"<code>{col}</code>",
            'Mértékegység': unit,
            'Érvényes (db)': f"{n_valid:,} ({pct:.1f}%)".replace(',', ' '),
            'Szerepe a kutatásban': role
        })

df_qual = pd.DataFrame(stats_rows).sort_values(['Kategória', 'Kitöltöttség (%)'], ascending=[True, True])

# 3. Strukturált, kategóriánként színezett kitöltöttségi ábra
fig2 = px.bar(
    df_qual,
    x='Kitöltöttség (%)',
    y='Változó megnevezése',
    color='Kategória',
    orientation='h',
    title=f'Kutatási Változók Kitöltöttségi Aránya Funkcionális Kategóriánként (N = {len(df):,} db)'.replace(',', ' '),
    labels={'Kitöltöttség (%)': 'Kitöltöttség aránya (%)', 'Változó megnevezése': 'Változó'},
    range_x=[0, 105],
    template=PLOTLY_TEMPLATE,
    height=850
)
fig2.update_layout(yaxis=dict(tickfont=dict(size=10)), margin=dict(l=220, r=30, t=50, b=50))
fig2.show()

# 4. Teljes HTML Adatszótár megjelenítése
df_dict = pd.DataFrame(dict_rows)
html_dict = "<div style='overflow-x:auto; margin: 20px 0; max-height: 480px; overflow-y: auto;'>" + df_dict.to_html(classes='table table-bordered table-striped table-hover', index=False, escape=False) + "</div>"
display(HTML("<b>Részletes Kutatási Adatszótár és Változóleírás:</b>" + html_dict))
display(HTML("<p style='color:#64748b; font-size:12px;'>Megjegyzés: a hálózati távolság-, izokrón- és POI-változók kitöltöttsége a garantáltan pontos almintán értelmezett (strukturális hiány, nem adathiány).</p>"))

### 4. Interaktív Adatszűrő és Validációs Panel

**📌 Cél és funkció:**
Az interaktív táblázat az adathalmaz nyers szintű, sormintás ellenőrzését teszi lehetővé: szűrés eladó/kiadó típusra és GIS-pontosságra. A szűrt nézet azonnal újraszámolja a lokális átlag- és mediánárakat, lehetővé téve a mikroszegmensek ad hoc vizsgálatát.

In [ ]:
# Valós idejű kliensoldali adathalmaz-szűrő és feltáró vezérlőpult
import json

# Mikroadat a kliensoldali motorhoz: a szűrt minta VALÓDI mediánjának és átlagának számításához
microdata = []
for r in df.itertuples(index=False):
    if pd.isna(r.price_huf) or pd.isna(r.nm_ar_huf) or pd.isna(r.alapterulet_nm):
        continue
    microdata.append({
        't': str(r.listing_type), 'vr': str(r.varosresz), 'pt': int(r.minta_garantalt_pontos),
        'p': float(r.price_huf), 'nm': float(r.nm_ar_huf), 'a': float(r.alapterulet_nm)
    })

vr_list = sorted(list(df['varosresz'].dropna().unique()))

html_nb00 = f'''
<div id="nb00_filter_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🔎 Interaktív Adathalmaz Szűrő & Mintavételi Vezérlőpult</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus szegmentáció és statisztikai összegzés a teljes kőbányai ingatlanpiaci mintán</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap:18px; margin-bottom:18px;">
    <!-- Típus és Pontosság szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:10px; font-size:13px; text-transform:uppercase;">1. Hirdetés Típusa:</div>
      <div style="display:flex; gap:12px; margin-bottom:14px;">
        <label style="font-size:13px; font-weight:600; color:#334155; cursor:pointer;">
          <input type="radio" name="nb00_type" value="all" checked onchange="filterNB00()"> Mindkettő
        </label>
        <label style="font-size:13px; font-weight:600; color:#2563eb; cursor:pointer;">
          <input type="radio" name="nb00_type" value="elado" onchange="filterNB00()"> Eladó lakások
        </label>
        <label style="font-size:13px; font-weight:600; color:#d97706; cursor:pointer;">
          <input type="radio" name="nb00_type" value="kiado" onchange="filterNB00()"> Kiadó lakások
        </label>
      </div>

      <div style="font-weight:700; color:#334155; margin-bottom:8px; font-size:13px; text-transform:uppercase;">2. Térbeli Pontosság:</div>
      <label style="display:flex; align-items:center; gap:8px; font-size:13px; font-weight:600; color:#059669; cursor:pointer;">
        <input type="checkbox" id="nb00_exact" onchange="filterNB00()"> Csak garantált pontos koordinátás (N={n_pontos})
      </label>
    </div>

    <!-- Városrészek szűrő -->
    <div style="background:#f8fafc; padding:14px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
        <div style="font-weight:700; color:#334155; font-size:13px; text-transform:uppercase;">3. Városrészek Kijelölése:</div>
        <div style="font-size:11px;">
          <a href="javascript:void(0)" onclick="toggleAllVR(true)" style="color:#2563eb; margin-right:8px; text-decoration:none; font-weight:600;">Mind</a>
          <a href="javascript:void(0)" onclick="toggleAllVR(false)" style="color:#dc2626; text-decoration:none; font-weight:600;">Töröl</a>
        </div>
      </div>
      <div id="nb00_vr_chips" style="display:flex; flex-wrap:wrap; gap:6px; max-height:110px; overflow-y:auto; padding:4px;">
        {"".join([f'<label style="font-size:11px; background:#ffffff; border:1px solid #cbd5e1; border-radius:14px; padding:3px 8px; cursor:pointer;"><input type="checkbox" class="nb00_vr_cb" value="{vr}" checked onchange="filterNB00()"> {vr}</label>' for vr in vr_list])}
      </div>
    </div>
  </div>

  <!-- KPI Rács -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:16px;">
    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Szűrt Elemek</div>
      <div id="nb00_res_count" style="font-size:24px; font-weight:800; color:#1d4ed8; margin:3px 0;">-- db</div>
      <div style="font-size:11px; color:#2563eb;">Kiválasztott mintaelemszám</div>
    </div>

    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Átlagos Kínálati Ár</div>
      <div id="nb00_res_price" style="font-size:24px; font-weight:800; color:#15803d; margin:3px 0;">-- M Ft</div>
      <div id="nb00_lbl_price_type" style="font-size:11px; color:#16a34a;">Vételár átlag</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">Fajlagos Négyzetméterár</div>
      <div id="nb00_res_nm" style="font-size:24px; font-weight:800; color:#7e22ce; margin:3px 0;">-- Ft/m²</div>
      <div style="font-size:11px; color:#9333ea;">Szűrt minta átlaga</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:12px 16px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Átlagos Alapterület</div>
      <div id="nb00_res_area" style="font-size:24px; font-weight:800; color:#b45309; margin:3px 0;">-- m²</div>
      <div style="font-size:11px; color:#d97706;">Fizikai lakásméret</div>
    </div>
  </div>

  <!-- Városrészi Összesítő Mini Táblázat -->
  <div style="overflow-x:auto;">
    <table id="nb00_stat_table" class="table table-bordered table-striped table-hover" style="width:100%; font-size:12px; margin-top:8px;">
      <thead>
        <tr style="background:#f1f5f9; color:#334155;">
          <th>Városrész</th>
          <th style="text-align:right;">Hirdetésszám</th>
          <th style="text-align:right;">Átlagár (M Ft)</th>
          <th style="text-align:right;">Átlag Ár/m²</th>
          <th style="text-align:right;">Átlag Terület</th>
        </tr>
      </thead>
      <tbody></tbody>
    </table>
  </div>
</div>

<script>
(function() {{
  const data = {json.dumps(microdata)};
  const med = arr => {{ if (!arr.length) return 0; const s = [...arr].sort((a,b)=>a-b); const m = Math.floor(s.length/2); return s.length % 2 ? s[m] : (s[m-1]+s[m])/2; }};
  const mean = arr => arr.length ? arr.reduce((a,b)=>a+b,0)/arr.length : 0;

  window.toggleAllVR = function(state) {{
    document.querySelectorAll('.nb00_vr_cb').forEach(cb => cb.checked = state);
    filter();
  }};

  function filter() {{
    const type = document.querySelector('input[name="nb00_type"]:checked').value;
    const exactOnly = document.getElementById('nb00_exact').checked;
    const selectedVR = new Set();
    document.querySelectorAll('.nb00_vr_cb:checked').forEach(cb => selectedVR.add(cb.value));

    const rows = data.filter(r => (type === 'all' || r.t === type) && (!exactOnly || r.pt === 1) && selectedVR.has(r.vr));
    const P = rows.map(r => r.p), NM = rows.map(r => r.nm), A = rows.map(r => r.a);
    const totalN = rows.length;
    const vrMap = {{}};
    for (let r of rows) {{
      if (!vrMap[r.vr]) vrMap[r.vr] = {{ n:0, P:[], NM:[], A:[] }};
      vrMap[r.vr].n++; vrMap[r.vr].P.push(r.p); vrMap[r.vr].NM.push(r.nm); vrMap[r.vr].A.push(r.a);
    }}

    document.getElementById('nb00_res_count').innerText = totalN.toLocaleString('hu-HU') + ' db';
    if (type === 'kiado') {{
      document.getElementById('nb00_res_price').innerText = Math.round(med(P)/1000).toLocaleString('hu-HU') + ' ezer Ft/hó';
      document.getElementById('nb00_lbl_price_type').innerText = 'Havi bérleti díj medián (átlag: ' + Math.round(mean(P)/1000).toLocaleString('hu-HU') + ' ezer)';
    }} else {{
      document.getElementById('nb00_res_price').innerText = (med(P)/1e6).toFixed(1) + ' M Ft';
      document.getElementById('nb00_lbl_price_type').innerText = 'Kínálati vételár medián (átlag: ' + (mean(P)/1e6).toFixed(1) + ' M Ft)';
    }}
    document.getElementById('nb00_res_nm').innerText = Math.round(med(NM)).toLocaleString('hu-HU') + ' Ft/m²';
    document.getElementById('nb00_res_area').innerText = med(A).toFixed(1) + ' m²';

    const tbody = document.querySelector('#nb00_stat_table tbody');
    let rowsHtml = '';
    const vrSorted = Object.keys(vrMap).sort((a,b) => vrMap[b].n - vrMap[a].n);
    for (let vr of vrSorted) {{
      const v = vrMap[vr];
      const pMed = type === 'kiado' ? (Math.round(med(v.P)/1000).toLocaleString('hu-HU') + ' ezer') : (med(v.P)/1e6).toFixed(1) + ' M';
      rowsHtml += `<tr>
        <td><b>${{vr}}</b></td>
        <td style="text-align:right;">${{v.n}} db</td>
        <td style="text-align:right;">${{pMed}} <span style="color:#94a3b8; font-size:10px;">(átl. ${{type === 'kiado' ? Math.round(mean(v.P)/1000).toLocaleString('hu-HU') : (mean(v.P)/1e6).toFixed(1)}})</span></td>
        <td style="text-align:right;">${{Math.round(med(v.NM)).toLocaleString('hu-HU')}} Ft <span style="color:#94a3b8; font-size:10px;">(átl. ${{Math.round(mean(v.NM)).toLocaleString('hu-HU')}})</span></td>
        <td style="text-align:right;">${{med(v.A).toFixed(1)}} m²</td>
      </tr>`;
    }}
    tbody.innerHTML = rowsHtml || '<tr><td colspan="5" style="text-align:center; color:#94a3b8;">Nincs találat a kiválasztott szűrőkkel.</td></tr>';
  }}

  window.filterNB00 = filter;
  setTimeout(filter, 50);
}})();
</script>
'''
display(HTML(html_nb00))